In [1]:
# This is a clone of Andrej Karpathy's micrograd from scratch

In [2]:
import math

In [1]:
# Value class for calculations + back prop
class Value:
    def __init__(self, data, _op="", _prev=()):
        self.data = data
        self.grad = 0.0
        self._op = _op
        self._prev = _prev
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, _op='+', _prev=(self, other))

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, _op ='*', _prev = (self, other))

        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, _op='**', _prev=(self,))

        def _backward():
            self.grad += (other.data * self.data ** (other.data - 1)) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), _op='exp', _prev=(self,))

        def _backward():
            self.grad += math.exp(self.data) * out.grad

        out._backward = _backward
        return out
    
    def tanh(self):
        val = (math.exp(self.data) - math.exp(-self.data))/(math.exp(self.data) + math.exp(-self.data))
        out = Value(val, _op='tanh', _prev=(self,))

        def _backward():
            self.grad += (1 - val**2) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0, _op="relu", _prev=(self,))

        def _backward():
            self.grad += (1 if self.data > 0 else 0) * out.grad

        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        return self * (other**-1)

    def backward(self):
        topo = []
        visited = set()
        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)
        build_topo(self)
        self.grad = 1.0
        
        for node in reversed(topo):
            node._backward()

        

In [5]:
import random

In [3]:
# Neuron class to initialize neurons with weights and biases
class Neuron:
    def __init__(self, nin):
        self.weights = [Value(random.gauss(0,1)) for i in range(nin)]
        self.bias = Value(random.gauss(0,1))

    def __call__(self, x):
        #forward function
        act = sum((wi * xi for wi, xi in zip(self.weights, x), self.bias)
        return act.tanh()

    def parameters(self):
        return self.weights + [self.bias]

_IncompleteInputError: incomplete input (3779760020.py, line 2)